In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

In [76]:
df = pd.read_csv('churn_data.csv')

In [77]:
df

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,churn_flag,name,country,state,gender,dob,complaint_date,escalations,csat_score,complaint_count
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,...,0,keshav,India,Maharashtra,Male,1982-04-12,NaN,NaN,NaN,NaN
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,1,raghav,India,Karnataka,Male,1995-11-23,2024-08-28,Y,10.0,2.0
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,...,0,lalita,India,Delhi,Female,1978-02-15,NaN,NaN,NaN,NaN
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaN,NaN,22.99,1725,...,0,mohan,India,Nagaland,Male,2001-08-30,NaN,NaN,NaN,NaN
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,1,mira,India,Delhi,Female,1990-05-05,2024-01-20,Y,20.0,1.0
5,0013-MHZWF,2022-06-18,Paid,2025-06-18,Standard,Annual,NaN,NaN,17.99,720,...,0,durga,NaN,Delhi,Female,1988-12-10,2025-03-18,N,90.0,1.0
6,0013-SMEOE,2021-09-30,Refferal,2024-09-30,Basic,Monthly,2024-11-15,Not enough content,8.99,230,...,1,mina,India,Meghalaya,Female,1976-09-21,2024-11-01,N,30.0,1.0
7,0014-BMAQU,2020-02-14,Organic,2025-02-14,Premium,Annual,NaN,NaN,22.99,1840,...,0,madan,India,Rajasthan,Male,1999-03-14,NaN,NaN,NaN,NaN
8,0015-UOCOJ,2023-07-22,Organic,2024-07-22,Standard,Monthly,NaN,NaN,13.99,240,...,0,maya,NaN,Kathmandu,Female,1985-07-07,NaN,NaN,NaN,NaN
9,0016-QLJIS,2022-04-03,Organic,2025-04-03,Basic,Annual,NaN,NaN,6.99,335,...,0,arjun,Nepal,Kathmandu,Male,1993-10-29,NaN,NaN,NaN,NaN


In [78]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='str')

In [79]:
churn_rate = df['churn_flag'].mean()*100
print("Churn Rate:",churn_rate.round(2), "%")

Churn Rate: 28.57 %


In [80]:
# Retention Rate
retention_rate = 100 - churn_rate
print("Retention Rate:",round(retention_rate,2),"%")

Retention Rate: 71.43 %


In [81]:
churn_by_plan = df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name = 'churn_rate_pct')

In [82]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='str')

In [83]:
revenue = df['monthly_charges'].sum().round(2)
print("Revenue: ", revenue)

Revenue:  395.79


In [84]:
# ARPU - avg rev per user
arpu = df['monthly_charges'].mean()
print('ARPU:', round(arpu,2))

ARPU: 18.85


In [85]:
df.groupby('state').agg(
    churn_percentage = ('churn_flag', lambda x: round(x.mean()*100,2)),
    total_revenue = ('monthly_charges', 'sum'),
    total_cust = ('customerid', 'count')
).reset_index().sort_values(by = 'total_revenue')

,state,churn_percentage,total_revenue,total_cust
1,Karnataka,100.00,20.98,2
2,Kathmandu,0.00,20.98,2
5,Nagaland,0.00,22.99,1
7,Telangana,50.00,30.98,2
6,Rajasthan,0.00,36.98,2
4,Meghalaya,66.67,42.97,3
3,Maharashtra,0.00,50.97,3
0,Delhi,25.00,52.96,4
8,Uttar Pradesh,0.00,115.98,2


In [86]:
df.groupby('subscription_type').agg(
    churn_percentage = ('churn_flag', lambda x : round(x.mean()*100,2)),
    total_revenue = ('monthly_charges', 'sum'),
    total_customers = ('customerid', 'count')
)

,churn_percentage,total_revenue,total_customers
subscription_type,,,
Organic,0.00,145.91,9
Paid,16.67,174.94,6
Refferal,83.33,74.94,6


In [94]:
cols = ['subscription_start_date', 'renewal_date', 'cancellation_date','complaint_date' ]

df[cols] = df[cols].apply(pd.to_datetime)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     str           
 1   subscription_start_date  21 non-null     datetime64[us]
 2   subscription_type        21 non-null     str           
 3   renewal_date             21 non-null     datetime64[us]
 4   plan_type                21 non-null     str           
 5   contract_type            21 non-null     str           
 6   cancellation_date        6 non-null      datetime64[us]
 7   cancellation_reason      6 non-null      str           
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
 11  churn_flag               21 non-null     int64         
 12  name                     21 non-null     str     

In [108]:
today = pd.Timestamp.today()


df['tenure days'] = np.where(
    df['cancellation_date'].notna(),
    (df['cancellation_date'] - df['subscription_start_date']).dt.days,
    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure days'].mean().round(2)
print("Average Tenure Days:", avg_tenure)

Average Tenure Days: 1558.71


In [115]:
#revenue at risk - Revenue loss by churned users 
revenue_at_risk = df.loc[df['churn_flag'] == 1,['monthly_charges']].sum()
print("Revenue at risk in (K) = ", revenue_at_risk)


Revenue at risk in (K) =  monthly_charges    73.94
dtype: float64


In [119]:
# Escalation Rate
EscalationRate = (df['escalations'] == 'Y').mean()*100
print("Escalation Rate = ", round(EscalationRate,2), "%")

Escalation Rate =  19.05 %


In [ ]:
df['dob'] = pd.to_datetime(df['dob'])
today = pd.Timestamp.today()
df['age'] = (today.year - df['dob'].dt.year)

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,country,state,gender,dob,complaint_date,escalations,csat_score,complaint_count,tenure days,age
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,India,Maharashtra,Male,1982-04-12,NaT,NaN,NaN,NaN,2037.0,44
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,India,Karnataka,Male,1995-11-23,2024-08-28,Y,10.0,2.0,1501.0,31
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,...,India,Delhi,Female,1978-02-15,NaT,NaN,NaN,NaN,1422.0,48
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,...,India,Nagaland,Male,2001-08-30,NaT,NaN,NaN,NaN,2712.0,25
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,India,Delhi,Female,1990-05-05,2024-01-20,Y,20.0,1.0,419.0,36


In [150]:
# Avg Complaint per user
avg_complaint = df['complaint_count'].sum()/ df['customerid'].nunique()
print("Average Complaint = ",round(avg_complaint,2), "%")

Average Complaint =  0.43 %


In [156]:
df['escalations'] = np.where(
    df['escalations'] == 'Y', 1,0
)

In [160]:
corr_df = df[['escalations', 'churn_flag']].dropna()
correlation = corr_df['escalations'].corr(corr_df['churn_flag'])*100
print("Correlation b/w Escalations & Churn_Flag:", round(correlation,2))

Correlation b/w Escalations & Churn_Flag: 76.7


In [164]:
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 200)

In [166]:
pd.reset_option('display.max_columns')
df.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,country,state,gender,dob,complaint_date,escalations,csat_score,complaint_count,tenure days,age
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,...,India,Maharashtra,Male,1982-04-12,NaT,0,NaN,NaN,2037.0,44
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,India,Karnataka,Male,1995-11-23,2024-08-28,1,10.0,2.0,1501.0,31
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,...,India,Delhi,Female,1978-02-15,NaT,0,NaN,NaN,1422.0,48
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,NaN,22.99,1725,...,India,Nagaland,Male,2001-08-30,NaT,0,NaN,NaN,2712.0,25
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,India,Delhi,Female,1990-05-05,2024-01-20,1,20.0,1.0,419.0,36


In [177]:
# df['churn_score'].head()
# df['churn_risk'] = pd.cut(
#     df['churn_score'],
#     bins = [0,50,70,100],
#     labels=['low','med','high']
# )

# df[['churn_score','churn_risk']].head()
# df[['churn_score','churn_risk']].tail()


conditions = [
    df['churn_score'] <= 50,
    (df['churn_score'] > 50 )& (df['churn_score'] <= 70),
    df['churn_score'] > 70
]

labels = ['low','med','high']

churn_risk = np.select(conditions,labels,default = 'unknown')


In [179]:
df.tail()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,state,gender,dob,complaint_date,escalations,csat_score,complaint_count,tenure days,age,churn_risk
16,0020-JDNXP,2022-01-19,Organic,2024-01-19,Premium,Annual,NaT,NaN,20.99,550,...,Meghalaya,Female,1994-08-19,NaT,0,NaN,NaN,1727.0,32,med
17,0021-IKXGC,2021-07-07,Paid,2025-07-07,Standard,Annual,NaT,NaN,13.99,840,...,Rajasthan,Female,2000-09-02,NaT,0,NaN,NaN,1923.0,26,low
18,0022-TCJCI,2023-09-14,Refferal,2024-09-14,Basic,Monthly,2024-09-14,Forgot to cancel trial,16.99,42,...,Telangana,Male,1983-12-30,2024-09-14,0,90.0,2.0,366.0,43,high
19,0023-HGHWL,2020-06-23,Organic,2025-06-23,Premium,Annual,NaT,NaN,22.99,1955,...,Uttar Pradesh,Male,1991-05-14,NaT,0,NaN,NaN,2302.0,35,low
20,0023-UYUPN,2022-12-31,Paid,2025-12-31,Standard,Monthly,NaT,NaN,13.99,790,...,Maharashtra,Female,1977-10-06,NaT,0,NaN,NaN,1381.0,49,low


# Visualisation

In [180]:
df_copy = df.copy()

In [ ]:
# Monthly Churn Rate
